# 04 · GenAI Tools Demo

* **Workshop:** AI for Actuaries — From Foundations to AI Agents
* **Session / Part:** S2.P2
* **Slides covered:** S2.P2.4, S2.P2.6, S2.P2.8
* **Author:** Dr Rohan Yashraj Gupta (FIA, FIAI), with Satya Sai Mudigonda and Sai Krishna Vadali
* **Workshop date:** 15 May 2026 · Hilton near Airport, Mumbai

## What this notebook does

Three reproducible Gemini API prompts that mirror Demos 1, 2, and 3 from the slide deck.

Demo 4 (Cursor IDE) is a live screen-share, not a notebook cell.

## Prerequisites

- Google account (for Colab)
- Gemini API key — set in **Colab Secrets** as `GEMINI_API_KEY` (free tier is sufficient)
- No local install required

## How to run

Top menu → **Runtime → Run all**. The first cell installs `google-genai`; the rest run without intervention.


## 0. Install

In [1]:
# Install google-genai. Pinned at 1.0+ — confirm against your Colab runtime
# at notebook freeze time.
!pip install -q google-genai

## 0.1 Standard imports and reproducibility

In [12]:
# === Standard imports ===
import json
import os
import datetime as dt
from pathlib import Path
from IPython.display import Markdown, display

# Reproducibility (LLM outputs are still non-deterministic by default —
# we set temperature=0 in calls below to get closer-to-stable results).
SEED = 42

# Notebook-specific imports below
# -------------------------------------------------------------
from google import genai
from google.genai import types

## 0.2 API setup

We read the Gemini API key from **Colab Secrets**. Add it in the left sidebar (key icon)
under the name `GOOGLE_API_KEY`. Local fallback: set the environment variable.

Pinning the model identifier matters — `gemini-2.5-flash` today is not the same model
in six months. Always pin in code.


In [19]:
# Read GOOGLE_API_KEY from Colab Secrets, fall back to env var.
try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GEMINI_API_KEY")
except (ImportError, Exception):
    if "GOOGLE_API_KEY" not in os.environ:
        raise RuntimeError(
            "GOOGLE_API_KEY is not set. "
            "Add it via Colab Secrets (left sidebar key icon) "
            "or set the env variable."
        )

# Pin the model. Confirm at notebook-finalisation time and update if the
# stable identifier has changed.
# MODEL_ID = "gemini-2.5-flash"
MODEL_ID = "gemini-3.1-flash-lite"

# Single shared client.
client = genai.Client()

print(f"API key loaded. Model pinned to: {MODEL_ID}")


API key loaded. Model pinned to: gemini-3.1-flash-lite


## 1. Demo 1 — risk-factor table for ABC Motor (GI)

**Slide:** S2.P2.4
**Goal:** Generate ten rating factors for an ABC Motor private car comprehensive
product, each tagged with directional impact and a one-line justification.
**Output format:** JSON

> ABC Insurer is hypothetical — see `00_personas_and_datasets.md`. Numbers are
> illustrative.


In [18]:
import json
from google import genai

# Create Gemini client
client = genai.Client()

# Select model
MODEL_ID = "gemini-3.1-flash-lite"

# Prompt
prompt = """
Generate 10 rating factors for an Indian private car insurance product.

For each factor return:
- factor
- direction ("increase" or "decrease")
- justification

Return ONLY valid JSON.
"""

# Generate structured output
response = client.models.generate_content(
    model=MODEL_ID,
    contents=prompt,
    config={
        "response_mime_type": "application/json",
    },
)

# Parse JSON
factors = json.loads(response.text)

# Clear visual separation
print("=" * 70)
print("📋 GEMINI MODEL RESPONSE")
print("=" * 70)

# Print formatted output
print(json.dumps(factors, indent=2))

print("\n" + "=" * 70)
print("END OF MODEL RESPONSE")
print("=" * 70)

📋 GEMINI MODEL RESPONSE
[
  {
    "factor": "Vehicle Age",
    "direction": "decrease",
    "justification": "As the vehicle ages, its Insured Declared Value (IDV) decreases, leading to lower premiums for own-damage coverage."
  },
  {
    "factor": "Geographic Zone",
    "direction": "increase",
    "justification": "Vehicles registered in Tier-1 cities (Zone A) are subject to higher premiums due to increased traffic density and higher accident/theft risk compared to rural areas."
  },
  {
    "factor": "Engine Cubic Capacity (CC)",
    "direction": "increase",
    "justification": "Higher engine capacity corresponds to higher IRDAI-mandated third-party liability premiums and increased repair costs."
  },
  {
    "factor": "No Claim Bonus (NCB)",
    "direction": "decrease",
    "justification": "A claim-free history rewards the policyholder with a cumulative discount on the premium, reducing the overall cost."
  },
  {
    "factor": "Vehicle Type (Fuel)",
    "direction": "increase",

## 2. Demo 2 — UW guidelines for ABC Term Life (Life)

**Slide:** S2.P2.6
**Goal:** Draft underwriting guidelines for an ABC Term Life product covering
issue ages 25–55 and sums assured up to ₹1 crore (Indian retail market).
**Output format:** Markdown, 300–400 words

> Output is a draft. Numerical thresholds (NoMed/TeleMed/FullMed bands etc.) must
> be matched against ABC Life's live underwriting matrix before publication.


In [16]:
from google import genai

# Create Gemini client
client = genai.Client()

# Select model
MODEL_ID = "gemini-3.1-flash-lite"

# Prompt
prompt = """
Draft underwriting guidelines for an Indian term life insurance product.

Include:
1. Eligibility
2. Medical underwriting
3. Financial underwriting
4. Smoker rules
5. PED handling

Assume:
- Entry age: 25–55
- Sum assured: up to ₹1 crore

Use simple markdown format.
"""

# Generate response
response = client.models.generate_content(
    model=MODEL_ID,
    contents=prompt,
)

# Clear visual separation
print("=" * 70)
print("📋 GEMINI MODEL RESPONSE")
print("=" * 70)

# Render markdown nicely in Colab
display(Markdown(response.text))

print("\n" + "=" * 70)
print("END OF MODEL RESPONSE")
print("=" * 70)

📋 GEMINI MODEL RESPONSE


These underwriting guidelines are designed for a standard term life insurance product in the Indian market, focusing on risk assessment for a Sum Assured (SA) of up to ₹1 crore.

---

### 1. Eligibility Criteria
*   **Entry Age:** 25 to 55 years (last birthday).
*   **Maximum Maturity Age:** 70 years.
*   **Policy Term:** 5 to 40 years.
*   **Residency:** Must be an Indian Resident (Indian citizens currently residing in India).
*   **Minimum Income:** Minimum annual income of ₹3 Lakhs per annum.
*   **Documentation:** PAN Card, Aadhaar Card, and latest Income Proof (ITR/Salary Slips) for SA > ₹50 Lakhs.

### 2. Medical Underwriting
To maintain cost-efficiency while managing risk, underwriting is tiered based on the Sum Assured and Age:

*   **Tier 1 (SA up to ₹25 Lakhs):** Purely Tele-Medical Underwriting (TMU). Self-declaration of health status.
*   **Tier 2 (SA ₹25 Lakhs to ₹75 Lakhs):** Mandatory Tele-Medical questionnaire + reports (HbA1c, Lipid Profile, Urine Routine) for applicants > 45 years.
*   **Tier 3 (SA ₹75 Lakhs to ₹1 Crore):** Mandatory Full Medical Examination (FME) including CBC, ECG, LFT, KFT, and HbA1c at an empanelled diagnostic center for all applicants.

### 3. Financial Underwriting
The Sum Assured is granted based on the "Human Life Value" (HLV) concept, typically capped at 15–20 times the annual income.

*   **Salaried:** Latest Form 16 / Salary Slips for the last 3 months.
*   **Self-Employed:** Last 3 years’ Income Tax Returns (ITR) with computation of income.
*   **Housewives/Non-earning individuals:** Generally not eligible for pure term products unless a specific "Spouse Cover" add-on is opted for (subject to husband’s income proof).
*   **Debt-to-Income:** Total coverage across all insurers should not exceed 20x the annual income.

### 4. Smoker Rules
The product uses "Smoking Status" as a primary rating factor for premiums.

*   **Definition:** Any usage of tobacco, cigarettes, cigars, gutkha, or nicotine patches within the last 12 months.
*   **Declaration:** Must be disclosed at the time of proposal.
*   **Rating:** 
    *   **Non-Smoker:** Standard premium rates.
    *   **Smoker:** Loading of 20%–40% over the standard premium.
*   **Verification:** Cotinine urine test may be mandated for higher Sum Assured categories or if medical reports indicate tobacco-related markers. Misrepresentation of smoking status leads to policy cancellation/claim rejection.

### 5. Pre-Existing Disease (PED) Handling
PEDs are evaluated based on severity, control, and duration of the condition.

*   **Minor conditions (e.g., controlled Hypertension/Type 2 Diabetes):** Acceptable with a "Medical Loading" (extra premium) if documented and stable for > 2 years.
*   **Major conditions (e.g., Cancer history, Chronic Kidney Disease, Heart Surgery):** Generally considered **Declined**.
*   **Non-disclosure:** Any PED not declared during the proposal stage is subject to the **Section 45 of the Insurance Act, 1938** (Incontestability Clause), allowing the insurer to repudiate claims if fraud is detected within the first 3 years.
*   **Standard Waiting Period:** None for death benefits; however, any PEDs must be declared at the point of inception to ensure the validity of the contract.

---

***Disclaimer:** These guidelines are for drafting purposes only and should be vetted by a qualified actuary and legal counsel to comply with IRDAI (Insurance Regulatory and Development Authority of India) regulations.*


END OF MODEL RESPONSE


## 3. Demo 3 — ABC Health board summary (Health)

**Slide:** S2.P2.8
**Goal:** Take a JSON dict of model results from a hospitalisation-frequency
model and produce a 200-word executive summary suitable for the ABC Health
board pack.
**Output format:** Markdown, ~200 words, saved to `/content/demo_3_output.md`.

The pattern: structured numbers in, structured prose out. No statistic in the
output that wasn't in the input.


In [17]:
import json
from google import genai
from IPython.display import Markdown, display

# Create Gemini client
client = genai.Client()

# Select model
MODEL_ID = "gemini-3.1-flash-lite"

# Example health insurance model results
model_results = {
    "model_name": "Hospitalisation Risk Model",
    "members_modelled": 120000,
    "model_accuracy_auc": 0.81,
    "high_risk_member_lift": 3.2,
    "top_risk_drivers": [
        "Member Age",
        "Diabetes History",
        "Previous Hospitalisation"
    ],
    "fairness_gap_percent": 1.8
}

# Prompt
prompt = f"""
Write an executive summary for the board of a health insurance company.

Use the following model results:
{json.dumps(model_results, indent=2)}

Requirements:
- Around 150 words
- Use simple business language
- Explain the model performance clearly
- Mention the top risk drivers
- Mention the fairness gap
- End with a recommendation
- Return markdown only
"""

# Generate response
response = client.models.generate_content(
    model=MODEL_ID,
    contents=prompt,
)

# Clear visual separation
print("=" * 70)
print("📋 GEMINI MODEL RESPONSE")
print("=" * 70)

# Render markdown nicely in Colab
display(Markdown(response.text))

print("\n" + "=" * 70)
print("END OF MODEL RESPONSE")
print("=" * 70)

📋 GEMINI MODEL RESPONSE


### Executive Summary: Hospitalisation Risk Model

We have successfully developed the "Hospitalisation Risk Model" to proactively identify members at the highest risk of future hospital admission. Covering a cohort of 120,000 members, the model demonstrates high predictive reliability with an AUC score of 0.81. Most importantly, it delivers a 3.2x lift in identifying high-risk members compared to baseline methods, allowing for more precise resource allocation.

Our analysis reveals that the primary drivers of hospitalisation risk are member age, diabetes history, and previous hospitalisations. We have also rigorously audited the model for bias; it currently maintains a low fairness gap of 1.8%, ensuring equitable outcomes across our member segments.

Given these strong results, we recommend immediate operational integration of the model. By embedding these insights into our existing care management workflows, we can intervene earlier, improve member health outcomes, and significantly reduce avoidable medical costs. We request approval to move into the pilot implementation phase next quarter.


END OF MODEL RESPONSE


## Wrap-up

You should now be able to:

- Call the Gemini API from Colab with a pinned model identifier and an API key loaded from Colab Secrets.
- Run a structured-output prompt that returns parseable JSON (Demo 1).
- Run a freeform-markdown prompt with explicit structure, length, and voice constraints (Demo 2).
- Feed a JSON of model results into a prompt and produce a board-ready prose summary that uses only the numbers you fed in (Demo 3).


**Where to next:** open `05_agents_intro.ipynb` for the minimal Agno + Gemini hello-agent that powers Part 3 of Session 2.

**Companion slides:** S2.P2.4 (Demo 1), S2.P2.6 (Demo 2), S2.P2.8 (Demo 3).

**Validation reminder:** The actuary owns the number, not the model. Before any output here goes near a regulatory deliverable, verify every figure against a real source.
